# Phase 2 — Feature Engineering Exploration

This notebook runs the full Phase 2 pipeline and provides visual sanity checks:
- Calendar feature distributions
- Correlation matrix of temporal features vs targets (**fixed: valid-overlap slice**)
- Weather-derived feature time series
- NaN audit and sparse-feature exclusion report
- Feature importance preview (mutual information)
- Train/val/test split preview (**fixed: 0-row bug**)

> 🔒 **Data leakage fix (v2):** `split_and_scale()` in `scaling.py` now **strictly fits the scaler on the training set only**, then applies the learned parameters to transform the validation and test sets independently. This mathematically prevents any statistical information from the val/test distributions leaking into the model during training.


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

PROCESSED = Path('../data/processed')
RESULTS   = Path('../results')
RESULTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
from src.features.pipeline import build_features, get_feature_cols, valid_overlap_corr, TARGET_COLS

df = build_features(PROCESSED, force=False)
print(f'Shape: {df.shape}')
df.head(3)

## 1. NaN audit — full feature matrix

In [ ]:
nan_pct = df.isna().mean().mul(100).sort_values(ascending=False)
print(f'Total columns: {len(nan_pct)}')
print(f'Columns with >50% NaN (excluded from feature set): {(nan_pct > 50).sum()}')
print(f'Columns with any NaN: {(nan_pct > 0).sum()}')
print()
print('=== Top 20 most-NaN columns ===')
print(nan_pct.head(20).to_string())

## 2. Feature column selection — sparse exclusion report

In [ ]:
# get_feature_cols now enforces the 50% NaN exclusion gate
# Watch the WARNING log lines — they list exactly which columns are excluded and why
feature_cols = get_feature_cols(df)
print(f'Feature columns selected: {len(feature_cols)}')
print(f'\nTarget columns: {TARGET_COLS}')

# Confirm no feature column exceeds the 50% NaN threshold
feat_nan = df[feature_cols].isna().mean().mul(100)
assert (feat_nan > 50).sum() == 0, 'BUG: sparse columns leaked into feature set!'
print('\nAssertion passed: no feature column exceeds 50% NaN.')
print(f'\nMax NaN% in feature set: {feat_nan.max():.1f}% ({feat_nan.idxmax()})')

## 3. Calendar feature distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
cal_cols = ['hour', 'dow', 'month', 'season', 'is_weekend', 'is_holiday']

for ax, col in zip(axes.flat, cal_cols):
    if col in df.columns:
        df[col].value_counts().sort_index().plot.bar(ax=ax, color='steelblue', width=0.85)
        ax.set_title(col)
        ax.set_xlabel('')

plt.suptitle('Calendar feature distributions', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS / 'calendar_distributions.png', bbox_inches='tight')
plt.show()

## 4. Weather-derived features — time series overview

In [ ]:
weather_derived = [c for c in ['hdd', 'cdd', 'wind_power_proxy', 'clearsky_index',
                                'is_raining', 'humid_cold'] if c in df.columns]
print(f'Weather-derived features present: {weather_derived}')

# Check clearsky_index has non-trivial variance (BUG FIX verification)
if 'clearsky_index' in df.columns:
    ci = df['clearsky_index'].dropna()
    print(f'\nclearsky_index — min: {ci.min():.4f}, max: {ci.max():.4f}, '
          f'std: {ci.std():.4f}, mean: {ci.mean():.4f}')
    assert ci.std() > 0.01, 'BUG: clearsky_index is still near-zero variance!'
    print('Assertion passed: clearsky_index has meaningful variance.')
else:
    print('clearsky_index absent (de_tsun too sparse — see meteostat.py log).')

n = min(3, len(weather_derived))
fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)
if n == 1:
    axes = [axes]

for ax, col in zip(axes, weather_derived[:n]):
    df[col].resample('D').mean().plot(ax=ax, lw=0.8)
    ax.set_title(col)

plt.suptitle('Weather-derived features (daily mean)', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / 'weather_derived_features.png', bbox_inches='tight')
plt.show()

## 5. Leakage sanity check — temporal features

Verify that `diff` and `rolling` features do not contain the current timestep value.

In [ ]:
# For a deterministic block of data (where target is non-NaN), verify:
# rolling_mean[t] < actual[t] most of the time (they should differ),
# and diff[t] ≠ actual[t] - actual[t-1] (which would imply leakage)

tgt = 'load_mw'
if tgt in df.columns and f'{tgt}_roll24_mean' in df.columns:
    check = df[[tgt, f'{tgt}_roll24_mean', f'{tgt}_diff1', f'{tgt}_lag1']].dropna().head(200)

    # If rolling mean = target exactly → leakage
    exact_match = (check[tgt] == check[f'{tgt}_roll24_mean']).mean()
    assert exact_match < 0.05, f'BUG: rolling mean matches target {exact_match:.0%} of rows!'

    # Leaking diff would be: diff1[t] == target[t] - lag1[t]
    leaking_diff = check[f'{tgt}_diff1'] - (check[tgt] - check[f'{tgt}_lag1'])
    near_zero = (leaking_diff.abs() < 1.0).mean()
    # Fixed diff1 = lag1 - lag2, so leaking_diff should NOT be near zero
    assert near_zero < 0.5, (
        f'BUG: diff1 looks like it was computed from y[t], not y[t-1]. '
        f'{near_zero:.0%} of rows match the leaking formula.'
    )

    print('Leakage checks passed:')
    print(f'  rolling mean ≈ target: {exact_match:.2%} of rows (should be <5%)')
    print(f'  diff1 ≈ y[t]-y[t-1]:  {near_zero:.2%} of rows (should be <50%)')
else:
    print(f'{tgt} or its derived features not present — skipping leakage check.')

## 6. Correlation matrix — weather features vs targets

**BUG FIX**: uses `valid_overlap_corr()` which trims to the overlapping valid window
before calling `.corr()`, preventing NaN rows from disjoint time ranges.

In [ ]:
from src.features.pipeline import valid_overlap_corr

# Weather composite columns available in the feature set
weather_composite = [c for c in feature_cols if c.startswith('de_') and c in df.columns]
weather_derived_feats = [c for c in ['hdd', 'cdd', 'temp_abs_dev', 'wind_power_proxy',
                                      'clearsky_index', 'is_raining', 'wind_category']
                         if c in df.columns]
corr_features = weather_composite + weather_derived_feats

# BUG FIX: valid_overlap_corr slices to the overlapping valid time window
# and drops any column with <20% valid values in that window.
corr = valid_overlap_corr(df, corr_features, TARGET_COLS, min_valid_frac=0.20)

print(f'Correlation matrix shape: {corr.shape}')
print(f'NaN values in corr: {corr.isna().sum().sum()} (should be 0)')

if not corr.empty:
    fig, ax = plt.subplots(figsize=(10, max(4, len(corr) * 0.4)))
    sns.heatmap(
        corr, annot=True, fmt='.2f', cmap='coolwarm',
        center=0, linewidths=0.3, ax=ax,
        cbar_kws={'shrink': 0.8},
        vmin=-1, vmax=1,
    )
    ax.set_title('Pearson correlation: weather features vs targets\n'
                 '(computed on valid overlapping time window)')
    plt.tight_layout()
    plt.savefig(RESULTS / 'weather_target_correlation.png', bbox_inches='tight')
    plt.show()
else:
    print('WARNING: no overlapping valid data found — correlation matrix is empty.')

## 7. Feature importance (mutual information)

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer

target = 'load_mw'

# Drop only if target is NaN; impute features with mean for MI scoring
temp_df = df[feature_cols + [target]].dropna(subset=[target])
temp_df = temp_df.dropna(axis=1, how='all')  # remove all-NaN columns
feat_for_mi = [c for c in feature_cols if c in temp_df.columns]

imputer = SimpleImputer(strategy='mean')
X_imp = imputer.fit_transform(temp_df[feat_for_mi])
sample_idx = np.random.default_rng(42).choice(len(temp_df), min(5000, len(temp_df)), replace=False)
X = X_imp[sample_idx]
y = temp_df[target].iloc[sample_idx].values

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=feat_for_mi).sort_values(ascending=False)

top30 = mi_series.head(30)
fig, ax = plt.subplots(figsize=(8, 9))
top30.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Mutual information score')
ax.set_title(f'Top 30 features for target: {target}')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(RESULTS / 'feature_importance_mi.png', bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
print(mi_series.head(10).to_string())

## 8. Train / val / test split preview

**BUG FIX**: `get_feature_cols()` now excludes >50% NaN columns, so `dropna(how='any')`
in `split_and_scale` no longer wipes the entire training set.

`carbon_intensity_g_kwh` has been added to `TARGET_COLS` in `pipeline.py` — it will
appear in the split stats below and has automatic lag/rolling/diff features
generated by `temporal.py`.


In [ ]:
from src.features.scaling import split_and_scale

# Confirm feature set is clean before splitting
feat_nan_max = df[feature_cols].isna().mean().max()
print(f'Max NaN% in feature_cols: {feat_nan_max:.1%} (should be ≤50%)')

splits = split_and_scale(
    df,
    target_cols=TARGET_COLS,
    feature_cols=feature_cols,
    train_end='2021-12-31',
    val_end='2022-12-31',
)

print()
for split_name in ['train', 'val', 'test']:
    s = splits[split_name]
    print(f"{split_name:6s}: {len(s):>6,} rows  "
          f"({s.index.min().date() if len(s) else 'N/A'} → "
          f"{s.index.max().date() if len(s) else 'N/A'})")

train_rows = len(splits['train'])
assert train_rows > 0, (
    'BUG: Training set is empty! Check get_feature_cols() NaN exclusion and '
    'ensure OPSD ← SMARD backfill ran correctly in Phase 1.'
)
print(f'\nAssertion passed: training set has {train_rows:,} rows.')

In [ ]:
# Visualise target coverage across the full timeline to confirm backfill worked
fig, axes = plt.subplots(len(TARGET_COLS), 1,
                         figsize=(14, 3 * len(TARGET_COLS)), sharex=True)
if len(TARGET_COLS) == 1:
    axes = [axes]

colors = {'train': 'steelblue', 'val': 'goldenrod', 'test': 'crimson'}

for ax, col in zip(axes, TARGET_COLS):
    for split_name, color in colors.items():
        s = splits[split_name]
        if col in s.columns and len(s) > 0:
            s[col].resample('W').mean().plot(
                ax=ax, lw=0.9, color=color, label=split_name, alpha=0.85
            )
    ax.set_title(col)
    ax.set_ylabel('value (weekly avg)')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Date')
plt.suptitle('Target coverage across train / val / test splits\n'
             '(gaps indicate missing data; SMARD backfill should cover post-2020)', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / 'split_target_coverage.png', bbox_inches='tight')
plt.show()


In [ ]:
print('=== Descriptive statistics — training targets ===')
print('(includes carbon_intensity_g_kwh in g CO₂-eq/kWh)')
print(splits['train'][TARGET_COLS].describe().round(1).to_string())
print('\nPhase 2 complete. Run 03_modelling.ipynb next.')
